# Nairobi flood project: Jermaine week 1
This notebook builds the prediction grid. It places evenly spaced points over north and east Nairobi, adds the same 9 features the model was trained on, and shows the points on a first map.

Run the cells from top to bottom, one at a time.

### 1. Install libraries

In [ ]:
!pip install -q geopandas pyogrio shapely rasterio requests scipy folium

### 2. Upload the geodatabase zip (FL20240426KEN_gdb.zip)

In [ ]:
from google.colab import files
files.upload()

### 3. Unzip it

In [ ]:
!unzip -o -q FL20240426KEN_gdb.zip

### 4. Save the feature script
This is the same script used to build the training data, so the grid features are calculated the same way.

In [ ]:
%%writefile add_features.py
# Adds terrain, land cover and waterway features to points.csv (run on Colab).
# Install once in Colab: !pip install -q rasterio geopandas shapely requests scipy
import sys
import numpy as np
import pandas as pd
from scipy.ndimage import uniform_filter

DEM_BASE = "https://copernicus-dem-30m.s3.amazonaws.com"
WC_BASE = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map"
MARGIN = 0.01


def slope_array(arr, pixel_x_m, pixel_y_m):
    gy, gx = np.gradient(arr, pixel_y_m, pixel_x_m)
    return np.degrees(np.arctan(np.hypot(gx, gy)))


def terrain_metrics(filled, pixel_x_m, pixel_y_m):
    smooth = uniform_filter(filled, size=3, mode="nearest")
    gy = np.gradient(smooth, pixel_y_m, axis=0)
    gx = np.gradient(smooth, pixel_x_m, axis=1)
    gyy = np.gradient(gy, pixel_y_m, axis=0)
    gxx = np.gradient(gx, pixel_x_m, axis=1)
    return {
        "local_relief_100": filled - uniform_filter(filled, size=7, mode="nearest"),
        "local_relief_500": filled - uniform_filter(filled, size=33, mode="nearest"),
        "concavity": (gxx + gyy) * 1000.0,
    }


def neighbourhood_share(mask, size):
    return uniform_filter(mask.astype("float32"), size=size, mode="nearest")


def sample_array(arr, c, a, f, e, lons, lats):
    cols = np.floor((lons - c) / a).astype(int)
    rows = np.floor((lats - f) / e).astype(int)
    ok = (rows >= 0) & (rows < arr.shape[0]) & (cols >= 0) & (cols < arr.shape[1])
    out = np.full(len(lons), np.nan)
    out[ok] = arr[rows[ok], cols[ok]]
    return out


def dem_tile_name(lat_floor, lon_floor):
    ns = "S" if lat_floor < 0 else "N"
    ew = "E" if lon_floor >= 0 else "W"
    return f"Copernicus_DSM_COG_10_{ns}{abs(lat_floor):02d}_00_{ew}{abs(lon_floor):03d}_00_DEM"


def add_elevation_slope(df):
    import rasterio
    from rasterio.merge import merge
    x0, x1 = df.lon.min() - MARGIN, df.lon.max() + MARGIN
    y0, y1 = df.lat.min() - MARGIN, df.lat.max() + MARGIN
    srcs = []
    for la in range(int(np.floor(y0)), int(np.floor(y1)) + 1):
        for lo in range(int(np.floor(x0)), int(np.floor(x1)) + 1):
            name = dem_tile_name(la, lo)
            url = f"/vsicurl/{DEM_BASE}/{name}/{name}.tif"
            print("opening", url)
            srcs.append(rasterio.open(url))
    arr, tr = merge(srcs, bounds=(x0, y0, x1, y1))
    arr = arr[0].astype("float64")
    arr[arr < -100] = np.nan
    mid_lat = (y0 + y1) / 2
    px = tr.a * 111320 * np.cos(np.radians(mid_lat))
    py = -tr.e * 110574
    filled = np.nan_to_num(arr, nan=np.nanmean(arr))
    layers = {"elevation": arr, "slope": slope_array(filled, px, py)}
    layers.update(terrain_metrics(filled, px, py))
    for name, layer in layers.items():
        df[name] = sample_array(layer, tr.c, tr.a, tr.f, tr.e, df.lon.values, df.lat.values)
    return df


def add_land_cover(df):
    import rasterio
    from rasterio.windows import Window, from_bounds
    lat_t = (np.floor(df.lat / 3) * 3).astype(int)
    lon_t = (np.floor(df.lon / 3) * 3).astype(int)
    for col in ("land_cover", "builtup_100", "builtup_500"):
        df[col] = np.nan
    for (la, lo), g in df.groupby([lat_t, lon_t]):
        name = f"ESA_WorldCover_10m_2021_v200_{'S' if la < 0 else 'N'}{abs(la):02d}{'E' if lo >= 0 else 'W'}{abs(lo):03d}_Map.tif"
        url = f"/vsicurl/{WC_BASE}/{name}"
        print("opening", url)
        m = 0.006
        with rasterio.open(url) as src:
            w = from_bounds(g.lon.min() - m, g.lat.min() - m, g.lon.max() + m, g.lat.max() + m, src.transform)
            win = Window(int(np.floor(w.col_off)), int(np.floor(w.row_off)), int(np.ceil(w.width)) + 2, int(np.ceil(w.height)) + 2)
            arr = src.read(1, window=win)
            tr = src.window_transform(win)
        built = arr == 50
        layers = {"land_cover": arr, "builtup_100": neighbourhood_share(built, 21), "builtup_500": neighbourhood_share(built, 101)}
        for col, layer in layers.items():
            df.loc[g.index, col] = sample_array(layer, tr.c, tr.a, tr.f, tr.e, g.lon.values, g.lat.values)
    return df


def add_river_distance(df):
    import requests
    import geopandas as gpd
    from shapely.geometry import LineString
    s, n = df.lat.min() - 0.03, df.lat.max() + 0.03
    w, e = df.lon.min() - 0.03, df.lon.max() + 0.03
    q = f'[out:json][timeout:180];way["waterway"~"^(river|stream|canal|drain|ditch)$"]({s},{w},{n},{e});out geom;'
    r = None
    for ep in ("https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"):
        r = requests.post(ep, data={"data": q}, headers={"User-Agent": "nairobi-flood-capstone/1.0", "Accept": "application/json"}, timeout=300)
        if r.ok:
            break
    r.raise_for_status()
    lines, kinds = [], []
    for el in r.json()["elements"]:
        if el.get("geometry") and len(el["geometry"]) > 1:
            lines.append(LineString([(p["lon"], p["lat"]) for p in el["geometry"]]))
            kinds.append(el.get("tags", {}).get("waterway", ""))
    print(len(lines), "waterway lines found")
    rivers = gpd.GeoDataFrame({"kind": kinds}, geometry=lines, crs=4326).to_crs(32737)
    pts = gpd.GeoDataFrame(df[["point_id"]], geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(32737)

    def nearest(sub):
        if len(sub) == 0:
            return None
        j = gpd.sjoin_nearest(pts, sub[["geometry"]], distance_col="d")
        return j[~j.index.duplicated()]["d"].reindex(df.index)

    df["dist_river"] = nearest(rivers)
    for col, kinds_in in (("dist_stream", ["river", "stream"]), ("dist_drain", ["canal", "drain", "ditch"])):
        d = nearest(rivers[rivers.kind.isin(kinds_in)])
        if d is not None:
            df[col] = d
    df["block_id"] = (pts.geometry.x // 1000).astype(int).astype(str) + "_" + (pts.geometry.y // 1000).astype(int).astype(str)
    return df


if __name__ == "__main__":
    src_file = sys.argv[1] if len(sys.argv) > 1 else "points.csv"
    df = pd.read_csv(src_file)
    df = add_elevation_slope(df)
    df = add_land_cover(df)
    df = add_river_distance(df)
    before = len(df)
    need = [c for c in ["elevation", "slope", "local_relief_100", "local_relief_500", "concavity",
                        "land_cover", "builtup_100", "builtup_500", "dist_river", "dist_stream", "dist_drain"]
            if c in df.columns]
    df = df.dropna(subset=need)
    water = (df.land_cover == 80).sum()
    df = df[df.land_cover != 80].copy()
    df["land_cover"] = df["land_cover"].astype(int)
    print("dropped missing:", before - len(df) - water, "| dropped permanent water:", water)
    df.to_csv("points_features.csv", index=False)
    print(df.shape)
    print(df.groupby("flooded")[[c for c in need if c != "land_cover"]].mean().round(3).T)

### 5. Find the area the satellite saw clearly
The grid only covers places the flood map could judge. Cloud covered places are removed.

In [ ]:
# Build the valid area from the UNOSAT layers
import geopandas as gpd
import shapely
from shapely.geometry import box

GDB = "FL20240426KEN.gdb"


def layer(name):
    return gpd.read_file(GDB, layer=name).to_crs(4326)


def union(gdf):
    geom = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
    return shapely.make_valid(geom)


seen = union(layer("PL_20240501_AnalysisExtent_Nairobi_Kiambu"))
cloud = union(layer("PL_20240501_CloudObstruction_Nairobi_Kiambu"))
valid = seen.intersection(box(36.65, -1.45, 37.10, -1.15)).difference(cloud)
print("valid area bounds:", valid.bounds)

### 6. Build the grid
Points are placed every 500 m and kept only if they fall inside the valid area.

In [ ]:
import numpy as np
import pandas as pd

GRID_M = 500  # spacing between grid points in metres


def grid_points(bounds, step_m):
    minx, miny, maxx, maxy = bounds
    mid_lat = (miny + maxy) / 2
    dlat = step_m / 110574
    dlon = step_m / (111320 * np.cos(np.radians(mid_lat)))
    lons = np.arange(minx, maxx, dlon)
    lats = np.arange(miny, maxy, dlat)
    lon_g, lat_g = np.meshgrid(lons, lats)
    return lon_g.ravel(), lat_g.ravel()


lon_g, lat_g = grid_points(valid.bounds, GRID_M)
inside = shapely.contains_xy(valid, lon_g, lat_g)

grid = pd.DataFrame({"lon": lon_g[inside], "lat": lat_g[inside]})
grid.insert(0, "point_id", range(1, len(grid) + 1))
print("grid points:", len(grid))
grid.head()

### 7. Check the grid

In [ ]:
# Confirm spacing and bounds look right
lat_steps = np.diff(np.sort(grid["lat"].unique()))
print("lat step in metres:", round(lat_steps.min() * 110574))
print("lon range:", grid["lon"].min().round(4), "to", grid["lon"].max().round(4))
print("lat range:", grid["lat"].min().round(4), "to", grid["lat"].max().round(4))
print("duplicate points:", grid.duplicated(["lon", "lat"]).sum())

### 8. Add the 9 model features
This step reads elevation, land cover and rivers from the web, so it can take a few minutes.

In [ ]:
# Calculate the features with the training script
import add_features as af

grid = af.add_elevation_slope(grid)
grid = af.add_land_cover(grid)
grid = af.add_river_distance(grid)
grid.shape

### 9. Keep the 9 features and save the grid
Points with missing values or on permanent water (land cover 80) are removed, as in the training data.

In [ ]:
# Clean the grid and keep the model columns
FEATURES = ["elevation", "slope", "local_relief_100", "local_relief_500", "concavity",
            "land_cover", "builtup_100", "builtup_500", "dist_river"]

before = len(grid)
grid = grid.dropna(subset=FEATURES)
grid = grid[grid["land_cover"] != 80].copy()
grid["land_cover"] = grid["land_cover"].astype(int)
print("removed:", before - len(grid), "| kept:", len(grid))

grid = grid[["point_id", "lon", "lat"] + FEATURES]
grid.to_csv("prediction_grid.csv", index=False)
grid.describe().T.round(2)

### 10. First map
The points have no colour for risk yet. Scores are added in week 2 when the tuned model arrives.

In [ ]:
# Plot the grid points on a map
import folium

centre = [grid["lat"].mean(), grid["lon"].mean()]
m = folium.Map(location=centre, zoom_start=11, tiles="CartoDB positron")

for row in grid.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=2,
        color="#222222",
        fill=True,
        fill_opacity=0.6,
        weight=0,
        tooltip=f"Point {row.point_id}: elevation {row.elevation:.0f} m",
    ).add_to(m)

m.save("grid_map.html")
m

### 11. Download the results

In [ ]:
files.download("prediction_grid.csv")
files.download("grid_map.html")